# 03 – Feature Engineering & Modelling

**Goal:** find the best model in a fair way.

1. Split into **train** and **test** – the test set is locked away until notebook 04.
2. Build a **baseline** (a model that always guesses the most common class).
3. Compare several models with **k-fold cross-validation** on the training set.
4. **Tune** the best model's hyper-parameters.

The same steps run automatically with `python -m ml.train`.

In [ ]:
import sys
from pathlib import Path

# Make the project root importable (notebooks/ is one level down)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from ml import config

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)

from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.model_selection import cross_validate, train_test_split
from sklearn.pipeline import Pipeline

from ml import train
from ml.features import add_features, build_preprocessor

In [ ]:
df = train.load_dataset()
X, y, input_cols = train.prepare_xy(df)
print("Input columns:", input_cols)
print("Target balance:", y.value_counts(normalize=True).round(3).to_dict())

## 1. Feature engineering

New features are created in `ml/features.py → add_features()`.
👉 TODO: add features that make sense for your problem.

In [ ]:
new_cols = [c for c in add_features(X).columns if c not in X.columns]
print("New features:", new_cols)
add_features(X)[new_cols].describe().T

## 2. Train / test split

In [ ]:
stratify = y if config.TASK_TYPE == "classification" else None
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=config.TEST_SIZE, random_state=config.RANDOM_STATE, stratify=stratify)
print(f"Train: {len(X_train)}  Test: {len(X_test)}")

## 3. Baseline – any real model must beat this

In [ ]:
dummy = DummyClassifier(strategy="stratified", random_state=0) if config.TASK_TYPE == "classification" else DummyRegressor()
baseline = Pipeline([("preprocess", build_preprocessor(X_train)), ("model", dummy)])
res = cross_validate(baseline, X_train, y_train, cv=config.CV_FOLDS, scoring=train.scoring())
pd.Series({m: res[f"test_{m}"].mean() for m in train.scoring()}).round(3)

## 4. Compare models with cross-validation

In [ ]:
cv_table = train.compare_models(X_train, y_train)
cv_table

In [ ]:
metric = config.PRIMARY_METRIC
ax = cv_table.sort_values(f"cv_{metric}_mean").plot.barh(
    x="model", y=f"cv_{metric}_mean", xerr=f"cv_{metric}_std", legend=False, color="#4472C4")
ax.set_xlabel(metric); ax.set_title("Model comparison (cross-validation)")
plt.show()

## 5. Tune the best model

In [ ]:
best_name = cv_table.iloc[0]["model"]
best_pipe, best_params, best_score = train.tune(best_name, X_train, y_train)
print(best_name, best_params, f"CV {metric} = {best_score}")

## 6. Train and save the final model

This runs the full pipeline (compare → tune → evaluate → save).

In [ ]:
report = train.run()

## Notes ✍️

👉 TODO: Which model won and why? Is it much better than the baseline?
Is there a big gap between models? What would you try next?